In [ ]:
!pip install -q transformers sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 18.6 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 2. ตั้งค่า Kaggle API ด้วย kaggle.json และดาวน์โหลดข้อมูล
# ==========================================
from google.colab import files
import os

print("กรุณาอัปโหลดไฟล์ kaggle.json ของคุณ:")
# จะมีปุ่ม Choose Files เด้งขึ้นมาให้กดอัปโหลด
uploaded = files.upload()

# สร้างโฟลเดอร์ .kaggle และย้ายไฟล์เข้าไป
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# เปลี่ยนสิทธิ์การอ่านไฟล์เพื่อความปลอดภัย (ถ้าไม่ทำ Kaggle จะฟ้อง Error)
!chmod 600 ~/.kaggle/kaggle.json

# ดาวน์โหลดและแตกไฟล์
!kaggle competitions download -c super-ai-engineer-ss-6-thai-language-image-captioning
!unzip -q super-ai-engineer-ss-6-thai-language-image-captioning.zip -d dataset/

กรุณาอัปโหลดไฟล์ kaggle.json ของคุณ:


Saving kaggle.json to kaggle.json
100% 1.75G/1.75G [00:09<00:00, 193MB/s]



In [15]:
!pip install -q transformers sentencepiece sacremoses

In [17]:
!pip install -q transformers sentencepiece sacremoses accelerate

**Cell 2: รัน Pipeline กู้ชีพ (ใช้เวลาประมาณ 5-10 นาทีสำหรับ 2,000 รูป)**

In [18]:
import torch
import glob
import os
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc

# ล้างขยะให้ชัวร์
torch.cuda.empty_cache()
gc.collect()

device_name = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. โหลด BLIP Large (FP16 + Auto RAM)
# ==========================================
print("👁️ Loading BLIP Large (Vision)...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    torch_dtype=torch.float16,
    device_map="auto" # ป้องกันแรมกระชาก
)

# ==========================================
# 2. โหลด NLLB 600M (FP16 + Auto RAM)
# ==========================================
print("🇹🇭 Loading NLLB 600M (Translator)...")
model_name = "facebook/nllb-200-distilled-600M" # กลับมาใช้ 600M รอดชัวร์!
nllb_tokenizer = AutoTokenizer.from_pretrained(model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

forced_bos_token_id = nllb_tokenizer.convert_tokens_to_ids("tha_Thai")

print("✅ Models loaded successfully! Starting TURBO generation...")

# ดึง Path รูปภาพ
test_image_dir = "dataset/test/"
test_image_paths = glob.glob(os.path.join(test_image_dir, "**", "*.jpg"), recursive=True)

submission_data = []

with torch.no_grad():
    for i, img_path in enumerate(test_image_paths):
        try:
            raw_image = Image.open(img_path).convert("RGB")

            # BLIP (ใส่ num_beams=4 ให้ตาดีขึ้น)
            inputs = blip_processor(raw_image, return_tensors="pt").to(device_name, torch.float16)
            out = blip_model.generate(**inputs, max_new_tokens=40, num_beams=4)
            en_caption = blip_processor.decode(out[0], skip_special_tokens=True)

            # NLLB (ใส่ num_beams=4 ให้แปลภาษาไทยสวยขึ้น)
            text_inputs = nllb_tokenizer(en_caption, return_tensors="pt").to(device_name)
            translated_tokens = nllb_model.generate(
                **text_inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=50,
                num_beams=4
            )
            th_caption = nllb_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

            # บันทึก
            img_id = os.path.basename(img_path).replace(".jpg", "")
            submission_data.append({"image_id": img_id, "caption": th_caption})

            if (i+1) % 50 == 0:
                print(f"[{i+1}/{len(test_image_paths)}] EN: {en_caption} -> TH: {th_caption}")

        except Exception as e:
            print(f"Error on {img_path}: {e}")

# เซฟเป็น CSV
df = pd.DataFrame(submission_data)
if not df.empty:
    df = df.sort_values("image_id")
    df.to_csv("submission_rescue_turbo.csv", index=False)
    print(f"🎉 เสร็จสิ้น! บันทึกไฟล์ 'submission_rescue_turbo.csv' จำนวน {len(df)} รูปเรียบร้อย")

👁️ Loading BLIP Large (Vision)...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🇹🇭 Loading NLLB 600M (Translator)...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Models loaded successfully! Starting TURBO generation...
[50/2000] EN: there are two men standing in front of a white and gold pagoda -> TH: มีชายสองคนยืนอยู่หน้าพากูดาสีขาวและทองคํา
[100/2000] EN: there is a large tree that is in the middle of the street -> TH: มีต้นไม้ขนาดใหญ่อยู่ตรงกลางถนน
[150/2000] EN: there is a close up of a yellow fruit on a plant -> TH: มีภาพใกล้เคียงของผลไม้เหลืองบนพืช
[200/2000] EN: there is a goat that is standing on the side of a hill -> TH: มีแพะที่ยืนอยู่บนฝั่งเขา
[250/2000] EN: trees are silhouetted against a purple and blue sky as the sun sets in the distance -> TH: ต้นไม้มีรูปร่างกับท้องฟ้าสีม่วงและสีฟ้า เมื่อดวงอาทิตย์ตกในระยะทาง
[300/2000] EN: there is a lizard that is sitting on top of a tree -> TH: มีกระเทียมที่นั่งบนต้นไม้
[350/2000] EN: there is a basket with a bunch of money in it on a table -> TH: มีกระเป๋าสตางค์ที่มีเงินอยู่ในกระเป๋าสตางค์บนโต๊ะ
[400/2000] EN: there is a brown and white cow standing in the dirt -> TH: มีวัวสีน้ําตาลและขาวยื